# 02 · ZeRO / FSDP: sharded data parallelism

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/distributed/02_zero_sharding.ipynb) &nbsp; 📖 Concept deck: [ZeRO and FSDP](https://nilesh-salpe.github.io/visual-ai/distributed/03-zero-fsdp/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** per-GPU memory for ZeRO stages 0–3, and a full ZeRO-3 training step (all-gather parameters, reduce-scatter gradients, sharded Adam) that matches single-GPU training exactly.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

r = np.random.default_rng(0)
D, N_RANK, B = 24, 4, 16                     # parameters, ranks, examples per rank
W_TRUE = r.normal(size=D)
DATA = [(r.normal(size=(B, D)), None) for _ in range(N_RANK)]
DATA = [(X, X @ W_TRUE + 0.1 * r.normal(size=B)) for X, _ in DATA]

def grad(W, X, y):
    """Gradient of mean squared error for a linear model."""
    return 2 * X.T @ (X @ W - y) / len(y)

def adam_update(w, g, m, v, t, lr=0.05, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1 - b1) * g; v = b2 * v + (1 - b2) * g * g
    mh, vh = m / (1 - b1 ** t), v / (1 - b2 ** t)
    return w - lr * mh / (np.sqrt(vh) + eps), m, v

def reference_training(steps):
    """One GPU on the full batch (the average of every rank's gradient)."""
    W = np.zeros(D); m = np.zeros(D); v = np.zeros(D)
    for t in range(1, steps + 1):
        g = np.mean([grad(W, X, y) for X, y in DATA], axis=0)
        W, m, v = adam_update(W, g, m, v, t)
    return W

## Task 1: Memory per GPU for each ZeRO stage

Mixed-precision Adam keeps 16 bytes per parameter: BF16 weights (2) + BF16 gradients (2) + FP32 master weights, momentum and variance (12).
* Stage 0 (plain data parallel): everything replicated: 16·P.
* Stage 1: shard the optimizer state: 2P + 2P + 12P/N.
* Stage 2: also shard gradients: 2P + 14P/N.
* Stage 3 (≈ FSDP full sharding): shard everything: 16P/N.

ZeRO reports "4x memory reduction" for stage 1, "8x" for stage 2 (as N grows) and reduction "linear with DP degree" for stage 3. Return bytes per GPU.

In [ ]:
def zero_bytes_per_gpu(P, N, stage):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_zero_bytes_per_gpu(f):
    P = 7e9
    assert np.isclose(f(P, 8, 0), 16 * P)
    assert np.isclose(f(P, 8, 3), 2 * P)
    assert np.isclose(f(P, 1e9, 0) / f(P, 1e9, 1), 4, rtol=1e-3), "stage 1 → 4× for large N"
    assert np.isclose(f(P, 1e9, 0) / f(P, 1e9, 2), 8, rtol=1e-3), "stage 2 → 8× for large N"
    assert np.isclose(f(P, 64, 0) / f(P, 64, 3), 64), "stage 3 → N×"

check("zero_bytes_per_gpu", _test_zero_bytes_per_gpu, zero_bytes_per_gpu)

## Task 2: A ZeRO-3 training loop

Each of the N ranks stores only its **shard** of the weights and of Adam's m and v (use `np.array_split(…, N)`). One step:
1. **All-gather** the shards into the full W (every rank needs it for its forward and backward pass).
2. Each rank computes the gradient on its **own data** `DATA[r]`.
3. **Reduce-scatter**: average the gradients across ranks, but each rank keeps only the slice matching its shard.
4. Each rank runs Adam on its shard only.

Return the final full W after `steps` steps. The test compares with `reference_training` (one GPU, full batch): they must match.

In [ ]:
def zero3_train(steps):
    W_shards = np.array_split(np.zeros(D), N_RANK)
    m_shards = [np.zeros_like(s) for s in W_shards]; v_shards = [np.zeros_like(s) for s in W_shards]
    # TODO: loop over steps: all-gather, local grads, reduce-scatter (average), sharded Adam
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_zero3_train(f):
    W = f(25)
    assert W.shape == (D,)
    assert np.allclose(W, reference_training(25), atol=1e-10), "sharded training must match single-GPU training exactly"

check("zero3_train", _test_zero3_train, zero3_train)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Memory per GPU for each ZeRO stage
def zero_bytes_per_gpu_solution(P, N, stage):
    return {0: 16 * P, 1: 4 * P + 12 * P / N, 2: 2 * P + 14 * P / N, 3: 16 * P / N}[stage]

check("zero_bytes_per_gpu (reference)", _test_zero_bytes_per_gpu, zero_bytes_per_gpu_solution)

In [ ]:
# @title Solution for Task 2: A ZeRO-3 training loop
def zero3_train_solution(steps):
    W_shards = np.array_split(np.zeros(D), N_RANK)
    m_shards = [np.zeros_like(s) for s in W_shards]; v_shards = [np.zeros_like(s) for s in W_shards]
    bounds = np.cumsum([0] + [len(s) for s in W_shards])
    for t in range(1, steps + 1):
        W = np.concatenate(W_shards)                                  # all-gather
        local = [grad(W, X, y) for X, y in DATA]                       # each rank: own data
        avg = np.mean(local, axis=0)                                   # reduce ...
        for k in range(N_RANK):                                        # ... scatter + sharded Adam
            g = avg[bounds[k]:bounds[k + 1]]
            W_shards[k], m_shards[k], v_shards[k] = adam_update(W_shards[k], g, m_shards[k], v_shards[k], t)
    return np.concatenate(W_shards)

check("zero3_train (reference)", _test_zero3_train, zero3_train_solution)

---
## Experiments
**Who fits where?** Per-GPU memory for model states (no activations) at several model sizes with 64 data-parallel GPUs of 80 GB. Stage 3 trades extra communication (an all-gather of parameters in both forward and backward) for memory.

In [ ]:
for P in [7e9, 70e9, 405e9]:
    row = '  '.join(f"stage {s}: {zero_bytes_per_gpu_solution(P, 64, s) / 1e9:7.1f} GB" for s in range(4))
    print(f"{P / 1e9:4.0f}B params, 64 GPUs → {row}")
print("\nCommunication per step (elements per GPU, ring): stage 0–2 ≈ 2·P (all-reduce or reduce-scatter + all-gather), stage 3 ≈ 3·P (one more parameter all-gather in backward).")

---
## Takeaways
* Plain data parallelism replicates 16 bytes per parameter on every GPU; ZeRO shards optimizer state (stage 1), gradients (2) and parameters (3).
* Sharding is exact: notebook results match single-GPU training bit for bit up to float error.
* Stage 3 / FSDP adds parameter all-gathers (≈1.5× the communication of plain DP), so it is overlapped with compute via prefetching; Llama 3 uses FSDP with resharding disabled after the forward pass to avoid one extra all-gather.